# 03 — Specificity Analysis

Compute cell-type specificity metrics for all genes and identify
transcripts uniquely expressed in each cell type.

**Metrics applied:**
1. **Tau specificity index** (≥ 0.85)
2. **On-target mean CPM** (≥ 1.0)
3. **On-target detection rate** (≥ 25%)
4. **Off-target detection rate** (≤ 5% in all other cell types)
5. **Fold-change** (≥ 10× vs. next-highest cell type)

**Inputs:** Preprocessed data from Notebook 02  
**Outputs:** `results/specificity_scores.csv`

In [ ]:
import sys
from pathlib import Path
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

from src.specificity import (
    compute_tau, compute_fold_change, score_specificity,
    compute_composite_score, query_cell_type,
)
from src.filters import apply_post_filters

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
CT_REPORT_DIR = RESULTS_DIR / "cell_type_reports"
RESULTS_DIR.mkdir(exist_ok=True)
CT_REPORT_DIR.mkdir(exist_ok=True)

## 1. Load Preprocessed Data

In [ ]:
# Raw CPM (for fold-change) and detection rate
mean_cpm = pd.read_parquet(DATA_DIR / "preprocessed_mean_cpm.parquet")
det_df = pd.read_parquet(DATA_DIR / "preprocessed_detection_rate.parquet")

# Log-transformed (for Tau — optional, score_specificity handles this internally)
mean_log = pd.read_parquet(DATA_DIR / "preprocessed_mean_log2cpm.parquet")

print(f"Loaded: {mean_cpm.shape[0]:,} genes × {mean_cpm.shape[1]} cell types")
print(f"Cell types: {list(mean_cpm.columns[:10])}...")

## 2. Compute Tau Specificity Index

Tau ranges from 0 (ubiquitous) to 1 (perfectly cell-type specific).

In [ ]:
tau = compute_tau(mean_log)

print(f"Tau statistics:")
print(tau.describe())
print(f"\nGenes with Tau ≥ 0.85: {(tau >= 0.85).sum():,}")
print(f"Genes with Tau ≥ 0.90: {(tau >= 0.90).sum():,}")
print(f"Genes with Tau ≥ 0.95: {(tau >= 0.95).sum():,}")

In [ ]:
from src.visualization import plot_tau_distribution

fig = plot_tau_distribution(
    tau, threshold=0.85,
    save_path=str(RESULTS_DIR / "tau_distribution.png"),
)
plt.show()

## 3. Run Full Specificity Scoring

This applies all metrics simultaneously and returns a ranked long-form table.

In [ ]:
results = score_specificity(
    mean_cpm=mean_cpm,
    det_df=det_df,
    tau_threshold=0.85,
    fc_threshold=10.0,
    on_target_det_min=0.25,
    off_target_det_max=0.05,
    on_target_expr_min=1.0,
)

print(f"Total gene-cell_type pairs with Tau ≥ 0.85: {len(results):,}")
print(f"Passing ALL thresholds: {results['all_pass'].sum():,}")
print(f"\nCell types with ≥1 specific gene: {results[results['all_pass']]['cell_type'].nunique()}")

In [ ]:
# Top hits overview
print("\nTop 20 most specific transcripts (passing all thresholds):")
display(results[results["all_pass"]].head(20))

## 4. Apply Post-Filters (Annotation Tiers)

Add annotation quality tiers. UniProt reviewed gene list and lncATLAS
data can be loaded here if available.

In [ ]:
# Optional: load UniProt reviewed gene list if available
# from src.filters import load_uniprot_reviewed_genes
# uniprot_path = PROJECT_ROOT / "data" / "references" / "uniprot_reviewed_human.tsv"
# uniprot_reviewed = load_uniprot_reviewed_genes(uniprot_path) if uniprot_path.exists() else None

results_filtered = apply_post_filters(
    results,
    uniprot_reviewed=None,  # Replace with uniprot_reviewed when available
    lncatlas_df=None,       # Load lncATLAS if filtering lncRNAs
    gene_length_map=None,   # Load from Ensembl BioMart if needed
    min_length=200,
)

print(f"After post-filtering: {len(results_filtered):,} gene-cell_type pairs")
print(f"Passing all thresholds: {results_filtered['all_pass'].sum():,}")

## 5b. Compute Composite Specificity Score

Combines Tau, fold-change, detection rate, expression level, and off-target detection
into a single weighted score in [0, 1] for ranking candidates within each cell type.

In [ ]:
results_filtered = compute_composite_score(
    results_filtered,
    weight_tau=0.35,
    weight_fc=0.25,
    weight_detection=0.20,
    weight_expression=0.10,
    weight_off_target=0.10,
)

print("Composite score added. Score distribution (all_pass=True genes):")
pass_scores = results_filtered[results_filtered["all_pass"]]["composite_score"]
print(pass_scores.describe().round(3))

print("\nTop 10 candidates across all cell types:")
display(results_filtered[results_filtered["all_pass"]].head(10)[
    ["gene", "cell_type", "composite_score", "tau", "fold_change", "detection_rate", "max_off_target_det"]
])

## 5. Per-Cell-Type Summary

In [ ]:
# Count specific genes per cell type
pass_only = results_filtered[results_filtered["all_pass"]]
ct_counts = pass_only["cell_type"].value_counts()

print(f"Cell types with specific transcripts: {len(ct_counts)}")
print(f"\nTop 20 cell types (most specific transcripts):")
print(ct_counts.head(20))

print(f"\nBottom 10 cell types (fewest specific transcripts):")
print(ct_counts.tail(10))

## 6. Example: Query a Specific Cell Type

## 7. Save Results

In [ ]:
results_filtered.to_csv(RESULTS_DIR / "specificity_scores.csv", index=False)
print(f"Saved {len(results_filtered):,} results to {RESULTS_DIR / 'specificity_scores.csv'}")

# Also save just the passing hits as a convenience
pass_only.to_csv(RESULTS_DIR / "specific_transcripts_all_pass.csv", index=False)
print(f"Saved {len(pass_only):,} passing hits to {RESULTS_DIR / 'specific_transcripts_all_pass.csv'}")

# Full results table (all genes passing Tau threshold)
results_filtered.to_csv(RESULTS_DIR / "specificity_scores.csv", index=False)
print(f"Saved {len(results_filtered):,} results → {RESULTS_DIR / 'specificity_scores.csv'}")

# All-pass hits only
pass_only = results_filtered[results_filtered["all_pass"]]
pass_only.to_csv(RESULTS_DIR / "specific_transcripts_all_pass.csv", index=False)
print(f"Saved {len(pass_only):,} passing hits → {RESULTS_DIR / 'specific_transcripts_all_pass.csv'}")

# Per-cell-type top-10 individual CSVs
TOP_N = 10
n_exported = 0
for ct in sorted(results_filtered["cell_type"].unique()):
    top = query_cell_type(results_filtered, ct, all_pass_only=True, top_n=TOP_N)
    if top.empty:
        # Fall back to top candidates even if not all_pass
        top = query_cell_type(results_filtered, ct, all_pass_only=False, top_n=TOP_N)
    if not top.empty:
        safe_name = ct.replace("/", "_").replace(" ", "_")
        top.to_csv(CT_REPORT_DIR / f"{safe_name}_top{TOP_N}.csv", index=False)
        n_exported += 1

print(f"\nPer-cell-type top-{TOP_N} CSVs saved → {CT_REPORT_DIR}/")
print(f"Cell types exported: {n_exported}")

# Combined top-5 per cell type in one table
top5_all = (
    results_filtered[results_filtered["all_pass"]]
    .groupby("cell_type", group_keys=False)
    .apply(lambda g: g.nlargest(5, "composite_score"))
    .reset_index(drop=True)
)
top5_all.to_csv(RESULTS_DIR / "top5_per_cell_type.csv", index=False)
print(f"Combined top-5-per-cell-type table saved → {RESULTS_DIR / 'top5_per_cell_type.csv'} ({len(top5_all)} rows)")